In [3]:
import pandas as pd
import gradio as gr
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile

sns.set_theme(style="whitegrid", palette="pastel")

# =========================
# LOAD DATA
# =========================

df = pd.read_csv(
    r"D:\EDA 4th\dashboard_project\data\Traffic_Collision_Data_from_2010_to_Present (2).csv"
)

df = df.drop_duplicates()
df.columns = df.columns.str.strip()

# =========================
# DATE HANDLING
# =========================

if "Date Occurred" in df.columns:
    df["Date Occurred"] = pd.to_datetime(df["Date Occurred"], errors="coerce")
    df = df[df["Date Occurred"].notna()]
    df["Year"] = df["Date Occurred"].dt.year
    df["Month"] = df["Date Occurred"].dt.month

# =========================
# CLEANING
# =========================

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("Unknown")
    else:
        df[col] = df[col].fillna(df[col].median())

# =========================
# AREA COLUMN
# =========================

area_col = None
for c in df.columns:
    if "AREA" in c.upper():
        area_col = c
        break

areas = sorted(df[area_col].dropna().unique().tolist()) if area_col else []

# =========================
# SAVE ZIP FUNCTION
# =========================

def save_zip(*figs):
    os.makedirs("charts", exist_ok=True)

    paths = []

    for i, fig in enumerate(figs, start=1):
        path = f"charts/chart_{i}.png"
        fig.savefig(path, bbox_inches="tight")
        paths.append(path)

    zip_path = "charts/all_charts.zip"

    with zipfile.ZipFile(zip_path, "w") as z:
        for p in paths:
            z.write(p)

    return zip_path

# =========================
# DASHBOARD FUNCTION
# =========================

def dashboard(start_date, end_date, categories, max_age, search_text):

    temp = df.copy()

    # DATE FILTER
    if "Date Occurred" in temp.columns:
        temp["Date Occurred"] = pd.to_datetime(temp["Date Occurred"], errors="coerce")

        if start_date:
            temp = temp[temp["Date Occurred"] >= pd.to_datetime(start_date, errors="coerce")]

        if end_date:
            temp = temp[temp["Date Occurred"] <= pd.to_datetime(end_date, errors="coerce")]

    # CATEGORY FILTER
    if area_col and categories:
        temp = temp[temp[area_col].isin(categories)]

    # AGE FILTER
    if "Victim Age" in temp.columns:
        temp = temp[temp["Victim Age"].notna()]
        temp = temp[temp["Victim Age"] <= max_age]

    # SEARCH FILTER
    if search_text:
        temp = temp[
            temp.astype(str).apply(
                lambda x: x.str.contains(search_text, case=False, na=False)
            ).any(axis=1)
        ]

    # =========================
    # KPI
    # =========================

    total = len(temp)
    avg_age = round(temp["Victim Age"].mean(), 2) if "Victim Age" in temp.columns else 0

    # =========================
    # 1 PIE CHART (FIXED)
    # =========================

    fig1, ax1 = plt.subplots()
    if "Victim Sex" in temp.columns:
        pie_data = temp["Victim Sex"].replace("Unknown", pd.NA).dropna()
        if len(pie_data) > 0:
            pie_data.value_counts().plot.pie(
                autopct="%1.1f%%",
                ax=ax1
            )
    ax1.set_title("Gender Distribution")

    # =========================
    # 2 COUNT PLOT (FIXED)
    # =========================

    fig2, ax2 = plt.subplots()
    if "Victim Sex" in temp.columns:
        count_data = temp["Victim Sex"].replace("Unknown", pd.NA).dropna()
        if len(count_data) > 0:
            order = count_data.value_counts().index
            sns.countplot(x=count_data, order=order, ax=ax2)
    ax2.set_title("Count Plot")

    # =========================
    # 3 VIOLIN PLOT (FIXED)
    # =========================

    fig3, ax3 = plt.subplots()
    if "Victim Sex" in temp.columns and "Victim Age" in temp.columns:
        clean = temp[["Victim Sex", "Victim Age"]].dropna()
        clean = clean[clean["Victim Sex"] != "Unknown"]
        if len(clean) > 0:
            sns.violinplot(x="Victim Sex", y="Victim Age", data=clean, ax=ax3)
    ax3.set_title("Violin Plot")

    # =========================
    # 4 HISTOGRAM
    # =========================

    fig4, ax4 = plt.subplots()
    if "Victim Age" in temp.columns:
        ax4.hist(temp["Victim Age"].dropna(), bins=25)
    ax4.set_title("Age Distribution")

    # =========================
    # 5 BAR CHART
    # =========================

    fig5, ax5 = plt.subplots()
    if area_col:
        temp[area_col].value_counts().head(10).plot(kind="bar", ax=ax5)
    ax5.set_title("Top Areas")

    # =========================
    # 6 LINE CHART
    # =========================

    fig6, ax6 = plt.subplots()
    if "Year" in temp.columns:
        temp.groupby("Year").size().plot(ax=ax6)
    ax6.set_title("Year Trend")

    # =========================
    # 7 SCATTER (SAFE FIX)
    # =========================

    fig7, ax7 = plt.subplots()
    lat_col = None
    for c in temp.columns:
        if "LAT" in c.upper():
            lat_col = c
            break

    if "Victim Age" in temp.columns and lat_col:
        clean = temp[[lat_col, "Victim Age"]].dropna()
        if len(clean) > 0:
            ax7.scatter(clean["Victim Age"], clean[lat_col])
    ax7.set_title("Scatter Plot")

    # =========================
    # 8 HEATMAP
    # =========================

    fig8, ax8 = plt.subplots()
    numeric = temp.select_dtypes(include=["int64", "float64"])
    if not numeric.empty:
        sns.heatmap(numeric.corr(), cmap="coolwarm", ax=ax8)
    ax8.set_title("Heatmap")

    # =========================
    # ZIP FILE
    # =========================

    zip_file = save_zip(fig1, fig2, fig3, fig4, fig5, fig6, fig7, fig8)

    return total, avg_age, temp.head(20), fig1, fig2, fig3, fig4, fig5, fig6, fig7, fig8, zip_file

# =========================
# UI
# =========================

with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("# 🚦 CLEAN FIXED TRAFFIC DASHBOARD")

    with gr.Row():

        with gr.Column(scale=1):
            start_date = gr.Textbox(label="Start Date")
            end_date = gr.Textbox(label="End Date")

            category = gr.CheckboxGroup(choices=areas, label="Area")

            age_slider = gr.Slider(0, 100, value=50, label="Max Age")

            search = gr.Textbox(label="Search")

            btn = gr.Button("Apply")

            download = gr.File(label="Download Charts ZIP")

        with gr.Column(scale=3):

            kpi1 = gr.Number(label="Total Records")
            kpi2 = gr.Number(label="Average Age")

            table = gr.Dataframe()

            with gr.Row():
                c1 = gr.Plot()
                c2 = gr.Plot()
                c3 = gr.Plot()
                c4 = gr.Plot()

            with gr.Row():
                c5 = gr.Plot()
                c6 = gr.Plot()
                c7 = gr.Plot()
                c8 = gr.Plot()

    btn.click(
        dashboard,
        inputs=[start_date, end_date, category, age_slider, search],
        outputs=[kpi1, kpi2, table, c1, c2, c3, c4, c5, c6, c7, c8, download]
    )

app.launch(share=True)

C:\Users\Azhar Computer\AppData\Local\Temp\ipykernel_12092\2761462656.py:221: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://6dd9ad4506dd8d9d6b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
# app.py
import gradio as gr
from filters import load_and_clean_data, apply_dashboard_filters
from charts import generate_all_plots, save_zip

# Global operational dataset synchronization execution
raw_df = load_and_clean_data()

# Dynamic identification mapping definitions
area_col = next((c for c in raw_df.columns if "AREA" in c.upper()), None)
unique_areas_list = sorted(raw_df[area_col].dropna().unique().tolist()) if area_col else []

def run_dashboard_pipeline(start_date, end_date, categories, max_age, search_text):
    """Symmetric unified calculation pipeline updating KPIs and all 10 plots synchronously."""
    # Process Filter Matrix logic mapping
    filtered_df = apply_dashboard_filters(raw_df, start_date, end_date, categories, max_age, search_text, area_col)
    
    # Calculate Performance Key Indicators
    total_incidents = len(filtered_df)
    avg_victim_age = round(filtered_df["Victim Age"].mean(), 2) if "Victim Age" in filtered_df.columns else 0
    preview_table = filtered_df.head(20)
    
    # Generate Visual Asset Matrix Maps
    charts = generate_all_plots(filtered_df, area_col)
    
    # Execute Local Storage deliverable packaging compression
    zip_export_path = save_zip(
        charts['plot_1'], charts['plot_2'], charts['plot_3'], charts['plot_4'], charts['plot_5'],
        charts['plot_6'], charts['plot_7'], charts['plot_8'], charts['plot_9'], charts['plot_10']
    )
    
    return (
        total_incidents, avg_victim_age, preview_table,
        charts['plot_1'], charts['plot_2'], charts['plot_3'], charts['plot_4'], charts['plot_5'],
        charts['plot_6'], charts['plot_7'], charts['plot_8'], charts['plot_9'], charts['plot_10'],
        zip_export_path
    )

# Building Layout Structuring Block Architecture Elements
with gr.Blocks(theme=gr.themes.Soft()) as web_dashboard_app:

    gr.Markdown("# 🚦 TRAFFIC COLLISION ANALYTICAL DISCOVERY INTERFACE")
    gr.Markdown("Exploratory Data Analysis framework reviewing geographical boundaries, victim demographics timelines, and feature correlations.")

    with gr.Row():
        # Left Panel Stack: Interaction Controller Navigation Filters Module
        with gr.Column(scale=1):
            gr.Markdown("### Navigation Search Parameters")
            ui_start = gr.Textbox(label="Start Date (YYYY-MM-DD)", placeholder="e.g., 2010-01-01")
            ui_end = gr.Textbox(label="End Date (YYYY-MM-DD)", placeholder="e.g., 2020-12-31")
            ui_category = gr.CheckboxGroup(choices=unique_areas_list, label="Area Locations Sector")
            ui_age = gr.Slider(minimum=0, maximum=120, value=100, label="Victim Age Threshold Limit")
            ui_search = gr.Textbox(label="Global Matrix Keyword Filter Search", placeholder="Enter query string data...")
            
            btn_execute = gr.Button("Apply Configuration Layer", variant="primary")
            download_module = gr.File(label="Download Compressed Visual Reports Packages (.ZIP)")

        # Right Panel Stack: Quantitative Performance Summary Cards, Preview Table & Grid Visualizer
        with gr.Column(scale=3):
            with gr.Row():
                kpi_total_count = gr.Number(label="Total Aggregated Records")
                kpi_average_metric = gr.Number(label="Average Victim Age Value")

            gr.Markdown("### Raw Historical Records Subsection Sample Viewer (Top 20 rows)")
            dataframe_viewer = gr.Dataframe()

            gr.Markdown("### Integrated Analytical Reporting Visual Charts Grid Matrix")
            # Symmetric Visual Presentation Grid Block Groups (10 Slots Layout configuration mapped)
            with gr.Row():
                slot_c1 = gr.Plot(label="1. Proportional Sex Breakdown")
                slot_c2 = gr.Plot(label="2. Incident Volume Profiles")
                slot_c3 = gr.Plot(label="3. Generative Distribution Profiles")
            with gr.Row():
                slot_c4 = gr.Plot(label="4. Density Age Spread Histogram")
                slot_c5 = gr.Plot(label="5. Spatial Volume Comparison")
                slot_c6 = gr.Plot(label="6. Trend Progression Matrix")
            with gr.Row():
                slot_c7 = gr.Plot(label="7. Spatial Core Variable Scatter")
                slot_c8 = gr.Plot(label="8. Relative Features Heatmap")
                slot_c9 = gr.Plot(label="9. Metric Ranges Detection Box")
            with gr.Row():
                slot_c10 = gr.Plot(label="10. Aggregated Monthly Area Trends")

    # Mapping Callback Routing Executions Setup
    inputs_pipeline_matrix = [ui_start, ui_end, ui_category, ui_age, ui_search]
    outputs_pipeline_matrix = [
        kpi_total_count, kpi_average_metric, dataframe_viewer,
        slot_c1, slot_c2, slot_c3, slot_c4, slot_c5, slot_c6, slot_c7, slot_c8, slot_c9, slot_c10,
        download_module
    ]
    
    btn_execute.click(
        fn=run_dashboard_pipeline,
        inputs=inputs_pipeline_matrix,
        outputs=outputs_pipeline_matrix
    )
    
    # Autoload metrics directly right away on browser deployment instantiation stage interface
    web_dashboard_app.load(
        fn=run_dashboard_pipeline,
        inputs=inputs_pipeline_matrix,
        outputs=outputs_pipeline_matrix
    )

if __name__ == "__main__":
    web_dashboard_app.launch(share=True)

ModuleNotFoundError: No module named 'filters'